# Queue: font-fill 10k + 20k (seed 42) + bandit 20k seeds 43 / 44 (4 runs)

Runs the training notebooks below one after another: font fill 10k, font fill 20k, then the bandit 20k seeds 43 and 44 (failed in queue_seeds_gap and queue_plain30k).

| arm | training set (public HF) | lines |
|---|---|---|
| font fill 10k: aggressive DiffusionPen words (verified), a word rejected 3 times drawn from the participant's unigram font, plan part 1 | `cyttic/heb-llmgen10k-diffpen-uni10aug-fontfill-hhdgeom`, shards 0-3 | 10,000 |
| font fill 20k: the same, plan parts 1 + 2 | `cyttic/heb-llmgen10k-diffpen-uni10aug-fontfill-hhdgeom`, shards 0-7 | 20,000 |
| bandit data mixer 20k | `cyttic/heb-llmgen20k-diffpenaug-hhdgeom` + `cyttic/heb-llmgen20k-uni10aug-hhdgeom` | 19,997 + 20,000 |

Each run gets its own folder `~/work/queue_fontfill/<name>/` and pushes its model + results to its own public HF repo.
A run whose HF repo already holds `results.json` is skipped, so **re-running this notebook resumes** the queue.
**A failed run's executed notebook (with the traceback) is uploaded to the public HF dataset `cyttic/a5000-queue-logs`**; the first cell also uploads the failed bandit notebooks of the two earlier queues.

Start on the server: `./run.sh diffpen_words/queue_fontfill.ipynb` (inside tmux).

In [ ]:
import os, json, time, traceback
import papermill as pm
from huggingface_hub import HfApi
NBS = [
 "exp39_fontfill10k_diffpenaug_hhdgeom_llmgen_colab.ipynb",
 "exp39_fontfill20k_diffpenaug_hhdgeom_llmgen_colab.ipynb",
 "exp39_bandit20k_diffpenaug_uni10aug_hhdgeom_llmgen_seed43_colab.ipynb",
 "exp39_bandit20k_diffpenaug_uni10aug_hhdgeom_llmgen_seed44_colab.ipynb"
]
HERE = os.getcwd(); BASE = os.path.expanduser('~/work/queue_fontfill'); os.makedirs(BASE, exist_ok=True)
api = HfApi()
LOGS = 'cyttic/a5000-queue-logs'   # PUBLIC dataset: executed notebooks of failed runs (tracebacks)
api.create_repo(LOGS, repo_type='dataset', private=False, exist_ok=True)

def push_log(path, name):
    try: api.upload_file(path_or_fileobj=path, path_in_repo=f'{name}.ipynb', repo_id=LOGS, repo_type='dataset', commit_message=f'failed run {name}'); print('log ->', f'{LOGS}/{name}.ipynb', flush=True)
    except Exception as e: print('log upload failed', repr(e)[:200], flush=True)

import glob   # the failed bandit runs of the earlier queues
for f in glob.glob(os.path.expanduser('~/work/queue_*/exp39_bandit*/*.ipynb')):
    push_log(f, f.split(os.sep)[-3] + '__' + os.path.basename(f)[:-6])

def out_repo(nb):
    src = ''.join(''.join(c['source']) for c in json.load(open(os.path.join(HERE, nb)))['cells'])
    return src.split('OUT_REPO = "', 1)[1].split('"', 1)[0]

def done(repo):
    try: return 'results.json' in api.list_repo_files(repo)
    except Exception: return False

status = {}
for i, nb in enumerate(NBS, 1):
    repo = out_repo(nb); name = nb.replace('_colab.ipynb', '')
    if done(repo): print(f'[{i}/{len(NBS)}] {name}: already on HF, skipped', flush=True); status[name] = 'skipped (done)'; continue
    wd = os.path.join(BASE, name); os.makedirs(wd, exist_ok=True); t0 = time.time()
    print(f'[{i}/{len(NBS)}] {name} -> {repo}  started {time.strftime("%H:%M")}', flush=True)
    try:
        pm.execute_notebook(os.path.join(HERE, nb), os.path.join(wd, nb), cwd=wd, log_output=True, kernel_name='python3')
        status[name] = f'ok ({(time.time() - t0) / 3600:.2f} h)' if done(repo) else 'finished, but results.json not on HF'
    except Exception as e:
        status[name] = 'FAILED: ' + repr(e)[:300]; traceback.print_exc(); push_log(os.path.join(wd, nb), 'queue_fontfill__' + name)
    print(f'[{i}/{len(NBS)}] {name}: {status[name]}', flush=True)
print('\n==== queue summary'); [print(f'{k:60s} {v}') for k, v in status.items()]

## Results on HF (greedy and beam 4 with the notebooks' own LM; the LM1 numbers for the paper are computed afterwards with eval_diffpen10k_vs_font10k.py)

In [ ]:
from huggingface_hub import hf_hub_download
for nb in NBS:
    repo = out_repo(nb)
    if not done(repo): print(f'{repo}: no results'); continue
    T = json.load(open(hf_hub_download(repo, 'results.json', force_download=True)))['final']['test_decoding']
    print(repo, {k.split(' (')[0]: round(100 * v['cer'], 2) for k, v in T.items()})